# 避難所候補算出支援ツール（sheltermatch）

要支援者一覧と避難所一覧の座標から、要支援者ごとに **距離が近い避難所候補（既定で上位3件）** を算出し、
必要であればハザード区域との位置関係も確認して、職員の最終判断のための資料（CSV）を作成するNotebookです。

**このツールが行うこと**
- 直線距離（`geopy.distance.geodesic`）による避難所候補の算出（候補提示。自動割当ではありません）
- 任意機能として、要支援者・避難所・両者を結ぶ直線とハザード区域（GeoJSON）との位置関係の確認

**このツールが行わないこと（重要）**
- 避難先・避難経路の自動決定
- 道路経路・通行可能性・坂道・高低差・災害時の道路状況の計算（ダイクストラ法や道路ネットワーク、Directions API等は使用しません）
- ハザード判定結果による候補避難所の自動除外・自動順位変更
- 独自の危険度スコアリング

距離順位とハザード判定は別々の情報として出力します。どの避難所を選ぶかは、出力結果を確認した **職員が判断** してください。

**個人情報の取り扱い**
- 実際の要支援者データ・ハザードデータはこのリポジトリにコミットしないでください（サンプルを追加する場合は完全な架空データを使用してください）。
- 出力CSVには住所・座標等の個人情報が含まれ得ます。取り扱いに注意してください。
- 住所→座標変換にはローカル辞書を使う **Jageocoder** のみを使用し、Google Maps API・OpenStreetMap Nominatim等の外部公開ジオコーディングサービスへ実住所を送信しません。

**Jageocoder辞書の準備（`ENABLE_GEOCODING = True` の場合のみ必要）**

沖縄県用のJageocoder辞書は、このリポジトリのGitHub Actions「沖縄県用Jageocoder辞書生成」で作成します
（Google Colab上では作成しません）。GitHub ActionsのArtifactは、ダウンロードすると外側にもう1つZIPが
かぶった状態になる点に注意してください。

```
1. GitHub Actions「沖縄県用Jageocoder辞書生成」を手動実行
2. ActionsのArtifact「okinawa-jageocoder-dictionary」をダウンロード
3. ダウンロードしたArtifact ZIPを一度展開する
4. 展開してできた中にある okinawa_jageocoder.zip を取り出す
5. sheltermatch.ipynb の実行時に、その okinawa_jageocoder.zip をアップロードする
   （Artifact ZIPそのものをアップロードしないこと）
```

## 使い方

1. 下の「利用者設定」を確認する
2. 「ランタイム → すべてのセルを実行」
3. 表示に従って
   - Jageocoder辞書ZIP（Artifact ZIPを展開して取り出した `okinawa_jageocoder.zip`）
   - 要支援者CSV
   - 必要に応じてハザードGeoJSON

   を選択する
4. 結果CSVを保存する

コードを読み込まなくても、この説明と各セルのprint出力だけで操作できます。

## 利用者設定

通常変更が必要な項目はこれだけです。値を確認・変更してから実行してください。

In [ ]:
# ===== 利用者設定 =====

# 住所から座標を取得する場合は True のままにしてください（後の手順でJageocoder辞書ZIPと
# 要支援者一覧のアップロードを求められます）。要支援者一覧CSVに緯度・経度が既に入っている
# 場合に限り、False に変更できます（その場合は辞書ZIPのアップロードは求められません）。
ENABLE_GEOCODING = True

# 要支援者・避難所とハザード区域（GeoJSON）との位置関係を確認する場合は True にしてください。
ENABLE_HAZARD_CHECK = False

# 要支援者ごとに算出する避難所候補の件数（避難所がこの件数未満の場合は存在する件数まで出力）
TOP_N = 3

# 避難所一覧の取得方法。"api"=BODIK Data APIから取得 / "csv"=CSVファイルをアップロード
# "api"で取得に失敗した場合は、自動的にCSVアップロードへ切り替わります。
SHELTER_SOURCE = "api"

if not isinstance(TOP_N, int) or TOP_N < 1:
    raise ValueError(f"TOP_N は1以上の整数を指定してください。現在の値: {TOP_N!r}")

if SHELTER_SOURCE not in ("api", "csv"):
    raise ValueError(f"SHELTER_SOURCE は 'api' または 'csv' を指定してください。現在の値: {SHELTER_SOURCE!r}")

print("利用者設定を読み込みました。")
print(f"  ENABLE_GEOCODING    = {ENABLE_GEOCODING}")
print(f"  ENABLE_HAZARD_CHECK = {ENABLE_HAZARD_CHECK}")
print(f"  TOP_N               = {TOP_N}")
print(f"  SHELTER_SOURCE      = '{SHELTER_SOURCE}'")

## 必要ライブラリの準備

Google Colabに標準で入っていないライブラリをインストールします。初回のみ数十秒かかることがあります。

In [ ]:
# jageocoderはGitHub Actionsで辞書を生成したときと同じバージョン(2.2.1.1)に固定する。
# 生成側と利用側のバージョンがずれると、辞書を正しく読み込めないことがある。
%pip install -q geopy "jageocoder==2.2.1.1" geopandas shapely

In [ ]:
import io
import os
import shutil
import zipfile

import numpy as np
import pandas as pd
import requests
from geopy.distance import geodesic

import geopandas as gpd
from shapely.geometry import Point, LineString

from google.colab import files

print("ライブラリの読み込みが完了しました。")

## Jageocoder辞書ZIPのアップロード

住所から座標を取得するため、GitHub Actions「沖縄県用Jageocoder辞書生成」で作成した辞書ZIP
（`okinawa_jageocoder.zip`）を1つアップロードしてください。アップロードされたZIPを展開し、
Jageocoderから実際に読み込めることを確認します。辞書として利用できない場合は、住所変換を開始せず
ここでエラーを表示して停止します。

`ENABLE_GEOCODING = False`（上の「利用者設定」）の場合はこのセルは何もせずスキップします
（アップロードは求められません）。

In [ ]:
JAGEOCODER_DB_DIR = "/content/jageocoder_db"


def safe_extract_zip(zip_path, extract_dir):
    """ZIPファイルをextract_dirへ安全に展開する。
    絶対パスや'..'を含むなど、展開先ディレクトリの外に出るエントリがあれば例外を送出する
    （パストラバーサル対策。展開自体は行わずに全エントリを先に検査してから実行する）。"""
    extract_dir_abs = os.path.abspath(extract_dir)
    with zipfile.ZipFile(zip_path) as zip_file:
        for member in zip_file.infolist():
            member_path = os.path.abspath(os.path.join(extract_dir_abs, member.filename))
            if member_path != extract_dir_abs and not member_path.startswith(extract_dir_abs + os.sep):
                raise RuntimeError(
                    f"ZIP内に不正なパスが含まれているため展開を中止しました（パストラバーサルの可能性）: {member.filename}"
                )
        zip_file.extractall(extract_dir_abs)


if ENABLE_GEOCODING:
    print("【Jageocoderの辞書ZIP】（GitHub Actions「沖縄県用Jageocoder辞書生成」で作成したZIPファイル）を選択してください。")
    uploaded_dictionary = files.upload()

    if len(uploaded_dictionary) != 1:
        raise RuntimeError("Jageocoderの辞書ZIPは1つだけ選択してください。")

    dictionary_zip_filename = list(uploaded_dictionary.keys())[0]
    dictionary_zip_bytes = uploaded_dictionary[dictionary_zip_filename]

    with open(dictionary_zip_filename, "wb") as zip_out:
        zip_out.write(dictionary_zip_bytes)

    if os.path.isdir(JAGEOCODER_DB_DIR):
        shutil.rmtree(JAGEOCODER_DB_DIR)
    os.makedirs(JAGEOCODER_DB_DIR, exist_ok=True)

    try:
        safe_extract_zip(dictionary_zip_filename, JAGEOCODER_DB_DIR)
    except zipfile.BadZipFile as error:
        raise RuntimeError(
            f"'{dictionary_zip_filename}' はZIPファイルとして展開できませんでした。"
            "GitHub Actions「沖縄県用Jageocoder辞書生成」のArtifactからダウンロードしたZIPファイルかどうか確認してください。"
            f" 詳細: {error}"
        )

    print(f"'{dictionary_zip_filename}' を '{JAGEOCODER_DB_DIR}' に展開しました。")

    import jageocoder

    try:
        jageocoder.init(db_dir=JAGEOCODER_DB_DIR)
    except Exception as error:
        raise RuntimeError(
            "展開した辞書をJageocoderから読み込めませんでした。"
            "GitHub Actions「沖縄県用Jageocoder辞書生成」のArtifactからダウンロードしたZIPファイルを"
            "正しくアップロードしているか確認してください。"
            f" 詳細: {error}"
        )

    print("Jageocoderで辞書を正常に読み込めることを確認しました。")
else:
    print("ENABLE_GEOCODING=False のため、Jageocoderの辞書ZIPアップロードをスキップしました。")

## 要支援者一覧CSVのアップロード

要支援者一覧CSVを選択してください。ファイル名は自由です。

In [ ]:
print("【要支援者一覧CSV】を選択してください。")
uploaded_residents = files.upload()

if len(uploaded_residents) != 1:
    raise RuntimeError("要支援者一覧CSVは1つだけ選択してください。")

residents_filename = list(uploaded_residents.keys())[0]
residents_bytes = uploaded_residents[residents_filename]
print(f"'{residents_filename}' を要支援者一覧として受け取りました。")

## 避難所一覧の取得

避難所一覧は上の「利用者設定」の `SHELTER_SOURCE` に従って取得します。

* `SHELTER_SOURCE = "api"`（既定）: BODIK Data API（CKANの `datastore_search`）から自治体標準ODSの
  避難所データを直接取得します。公開データの読み取りのみのためAPIキーは使用しません。
  取得に失敗した場合（通信エラー・レスポンス異常・0件など）は、Notebookを停止せずCSVアップロードへ
  自動的に切り替えます。
* `SHELTER_SOURCE = "csv"`: 避難所一覧CSVをブラウザから選択してアップロードします。

いずれの方法で取得しても、以降の列名変換（`名称`→`name` 等）・災害種別情報の処理・距離計算は同一です。

In [ ]:
BODIK_BASE_URL = "https://data.bodik.jp"
BODIK_RESOURCE_ID = "3132a0a4-f522-4b2d-bf18-f106d8b3a5ae"  # 糸満市 指定緊急避難場所データセット


def fetch_shelters_from_bodik(base_url, resource_id, page_size=1000):
    """BODIKのCKAN Data API(datastore_search)から避難所データを全件取得し、DataFrameで返す。
    total/offsetでページングして全件取得する。取得できない場合は例外を発生させ、
    呼び出し側でCSVアップロードへフォールバックする。"""
    endpoint = f"{base_url}/api/action/datastore_search"
    records = []
    offset = 0
    total = None

    while True:
        response = requests.get(
            endpoint,
            params={"resource_id": resource_id, "limit": page_size, "offset": offset},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()

        if not payload.get("success"):
            raise RuntimeError("CKAN APIレスポンスが success=false を返しました。")

        result = payload.get("result")
        if result is None or "records" not in result:
            raise RuntimeError("CKAN APIレスポンスに result.records が含まれていません。")

        page_records = result["records"]
        records.extend(page_records)

        if total is None:
            total = result.get("total", len(page_records))

        offset += len(page_records)
        if len(page_records) == 0 or offset >= total:
            break

    if len(records) == 0:
        raise RuntimeError("BODIK APIの取得結果が0件でした。")

    return pd.DataFrame(records)


shelters_raw = None
shelters_bytes = None

if SHELTER_SOURCE == "api":
    try:
        shelters_raw = fetch_shelters_from_bodik(BODIK_BASE_URL, BODIK_RESOURCE_ID)
        print(f"BODIK APIから避難所一覧を{len(shelters_raw)}件取得しました。")
    except Exception as error:
        print("BODIK APIから避難所一覧を取得できませんでした。")
        print("CSVファイルから読み込みます。")
        print(f"（詳細: {error}）")

if shelters_raw is None:
    print("【避難所一覧CSV】を選択してください。")
    uploaded_shelters = files.upload()

    if len(uploaded_shelters) != 1:
        raise RuntimeError("避難所一覧CSVは1つだけ選択してください。")

    shelters_filename = list(uploaded_shelters.keys())[0]
    shelters_bytes = uploaded_shelters[shelters_filename]
    print(f"'{shelters_filename}' を避難所一覧として受け取りました。")

## CSV読込・入力チェック

文字コードを自動判定して読み込みます（UTF-8 BOM付き → CP932 → UTF-8 の順で試行）。
続いて座標列（`latitude` / `longitude`）が数値として扱えるか、緯度が-90〜90、経度が-180〜180の範囲かを確認します。
座標が欠損・不正な要支援者の行があっても削除せず、後続の距離計算のみ対象外とします。
`ENABLE_GEOCODING = True` の場合は、`latitude` / `longitude` 列が無く `address` 列のみのCSVも読み込めます
（次のセクションでジオコーディングして座標を補完します）。

避難所一覧CSVが自治体標準オープンデータセット(ODS)形式（例:
[避難場所データセットの例](https://data.bodik.jp/dataset/472107_evacuation_space)）の場合、
日本語列名（`名称`→`name`、`緯度`→`latitude`、`経度`→`longitude`）を自動的に内部標準列名へ変換します。
従来の `name` / `latitude` / `longitude` 形式のCSVもそのまま利用できます。
`災害種別_` で始まる列（例: `災害種別_洪水`、`災害種別_崖崩れ、土石流及び地滑り` 等）がある場合は、
その避難所がどの災害種別に対応しているかを示す参考情報として保持し、避難所候補ごとにCSVへ出力します
（列名は固定せず、実データに存在するものをすべて認識します）。
自治体標準ODSの仕様では災害種別列の値に `1`(対応済み) / `2`(2階以上であれば対応済み) / 空欄(未対応) の
区別があり、この区別を失わずに `候補_対応済み` / `候補_2階以上であれば対応済み` 等の形で保持します
（`1`/`2`/空欄以外の想定外の値は「対応している」と推測せず、値をそのまま `(要確認)` として保持します）。
これは指定緊急避難場所としての対応種別情報であり、後述するGeoJSONによるハザード判定とは別の情報です。
両者を混同したり、対応種別によって距離順位・候補の自動除外を行うことはありません。

In [ ]:
def read_csv_auto(file_bytes, label):
    """UTF-8(BOM付き) → CP932 → UTF-8 の順で読み込みを試み、成功したDataFrameを返す。"""
    encodings = [
        ("utf-8-sig", "UTF-8 (BOM付き)"),
        ("cp932", "CP932 (Shift-JIS系)"),
        ("utf-8", "UTF-8"),
    ]
    last_error = None
    for encoding, encoding_label in encodings:
        try:
            df = pd.read_csv(io.BytesIO(file_bytes), encoding=encoding)
            print(f"[{label}] {encoding_label} として読み込みました。（{len(df)}行）")
            return df
        except (UnicodeDecodeError, UnicodeError) as error:
            last_error = error
            continue
    raise ValueError(
        f"[{label}] 文字コードを判定できませんでした。"
        "UTF-8(BOM付き)・CP932・UTF-8のいずれでも読み込めません。"
        "Excel等での保存時の文字コードを確認してください。"
        f" 詳細: {last_error}"
    )


def ensure_coordinate_columns(df, label):
    """latitude/longitude列が無い場合、ENABLE_GEOCODING=True かつ address列があれば
    ジオコーディング対象として空の座標列を追加する。それ以外は列不足として停止する。"""
    df = df.copy()
    if "latitude" in df.columns and "longitude" in df.columns:
        return df

    if ENABLE_GEOCODING and "address" in df.columns:
        df["latitude"] = np.nan
        df["longitude"] = np.nan
        print(f"[{label}] latitude/longitude列が無いため、address列からのジオコーディング対象として空の座標列を追加しました。")
        return df

    missing = [c for c in ("latitude", "longitude") if c not in df.columns]
    hint = "" if ENABLE_GEOCODING else "（ENABLE_GEOCODING=Trueにするとaddress列のみでも処理できます）"
    raise ValueError(f"[{label}] 必須列が見つかりません: {', '.join(missing)}{hint}")


# 避難所CSVの列名エイリアス（自治体標準ODS等の日本語列名 → 内部標準列名）
SHELTER_COLUMN_ALIASES = {
    "name": ["名称"],
    "latitude": ["緯度"],
    "longitude": ["経度"],
}

# 避難所の災害種別列の接頭辞（この接頭辞で始まる列を動的にすべて認識する）
DISASTER_TYPE_COLUMN_PREFIX = "災害種別_"

# 自治体標準ODSの災害種別列の値と対応区分の対応表（BODIKの指定緊急避難場所データセット仕様）
# 1=対応済み、2=2階以上であれば対応済み、空欄=未対応。これ以外の値は独自に「対応」と推測しない。
DISASTER_SUPPORT_LABELS = {
    "1": "対応済み",
    "1.0": "対応済み",
    "2": "2階以上であれば対応済み",
    "2.0": "2階以上であれば対応済み",
}


def normalize_shelter_columns(df, label):
    """自治体標準ODS等で使われる日本語列名を、内部標準列名(name/latitude/longitude)へ変換する。
    既に標準列名がある場合はそちらを優先し、変換しない。使用した列名の対応表も返す。"""
    df = df.copy()
    used_columns = {}
    for standard_col, aliases in SHELTER_COLUMN_ALIASES.items():
        if standard_col in df.columns:
            used_columns[standard_col] = standard_col
            continue
        for alias in aliases:
            if alias in df.columns:
                df = df.rename(columns={alias: standard_col})
                used_columns[standard_col] = alias
                break

    renamed = {std: orig for std, orig in used_columns.items() if orig != std}
    if renamed:
        mapping_text = ", ".join(f"{orig}→{std}" for std, orig in renamed.items())
        print(f"[{label}] 自治体標準ODS等の日本語列名を自動変換しました: {mapping_text}")

    return df, used_columns


def detect_disaster_type_columns(df):
    """列名が '災害種別_' で始まる列を、対応する災害種別情報として動的に検出する。
    特定の災害種別名の一覧に固定せず、実データに存在する列をそのまま採用する。"""
    return [c for c in df.columns if c.startswith(DISASTER_TYPE_COLUMN_PREFIX)]


def classify_disaster_support(value):
    """自治体標準ODSの災害種別列の値を解釈し、対応区分のラベルを返す（未対応ならNone）。
    '1'=対応済み、'2'=2階以上であれば対応済み、空欄=未対応という標準仕様に従う。
    それ以外の想定外の値は独自に「対応している」と推測せず、値をそのまま保持して要確認として返す。"""
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text == "":
        return None
    if text in DISASTER_SUPPORT_LABELS:
        return DISASTER_SUPPORT_LABELS[text]
    return f"{text}(要確認)"


def build_disaster_support(df, disaster_columns):
    """行ごとに対応している災害種別とその対応区分を ';' 区切りでまとめたSeriesを返す。
    例: '洪水:対応済み;高潮:2階以上であれば対応済み'。未対応(空欄)の種別は含めない。
    災害種別列が無ければ全行NaN。"""
    if not disaster_columns:
        return pd.Series([np.nan] * len(df), index=df.index, dtype="object")

    prefix_len = len(DISASTER_TYPE_COLUMN_PREFIX)

    def row_support(row):
        parts = []
        for col in disaster_columns:
            label = classify_disaster_support(row[col])
            if label is not None:
                parts.append(f"{col[prefix_len:]}:{label}")
        return ";".join(parts)

    return df.apply(row_support, axis=1)


residents_raw = read_csv_auto(residents_bytes, "要支援者一覧")

# 避難所一覧はBODIK APIから取得済み(shelters_raw is not None)の場合はそれを使い、
# CSVアップロード(APIフォールバック含む)の場合のみここでCSVを読み込む
if shelters_raw is None:
    shelters_raw = read_csv_auto(shelters_bytes, "避難所一覧")

# 避難所一覧は座標列の存否を確認する前に、自治体標準ODS等の日本語列名を内部標準列名へ変換する
shelters_raw, shelter_used_columns = normalize_shelter_columns(shelters_raw, "避難所一覧")
shelter_disaster_columns = detect_disaster_type_columns(shelters_raw)
shelters_raw["_disaster_support"] = build_disaster_support(shelters_raw, shelter_disaster_columns)
HAS_DISASTER_TYPE_COLUMNS = len(shelter_disaster_columns) > 0

residents_raw = ensure_coordinate_columns(residents_raw, "要支援者一覧")
shelters_raw = ensure_coordinate_columns(shelters_raw, "避難所一覧")

if "name" not in shelters_raw.columns:
    raise ValueError("[避難所一覧] 名称列が見つかりません: name または 名称 の列が必要です。")

print(
    f"[避難所一覧] 避難所件数: {len(shelters_raw)}件 / "
    f"名称列: '{shelter_used_columns.get('name', 'name')}' / "
    f"緯度列: '{shelter_used_columns.get('latitude', 'latitude')}' / "
    f"経度列: '{shelter_used_columns.get('longitude', 'longitude')}' / "
    f"認識した災害種別列数: {len(shelter_disaster_columns)}件"
)

display(residents_raw.head())
display(shelters_raw.head())

In [ ]:
def is_valid_coordinate(lat, lon):
    """緯度・経度が数値として有効な範囲かどうかを返す（欠損・範囲外はFalse）。
    距離計算・ハザード判定など、1点ずつ座標を扱う関数から共通で利用する。"""
    if pd.isna(lat) or pd.isna(lon):
        return False
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)


def parse_and_validate_coordinates(df, label):
    """latitude/longitude列を数値化し、有効な座標かどうかの真偽値Seriesを返す。"""
    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")
    valid = lat.notna() & lon.notna() & lat.between(-90, 90) & lon.between(-180, 180)
    invalid_count = int((~valid).sum())
    if invalid_count:
        print(f"[{label}] 座標が欠損・不正な行が {invalid_count}件あります（全{len(df)}行中）。")
    return lat, lon, valid


residents = residents_raw.copy()
residents["latitude"], residents["longitude"], residents_coord_valid = parse_and_validate_coordinates(
    residents, "要支援者一覧"
)

shelters = shelters_raw.copy()
shelters["latitude"], shelters["longitude"], shelters_coord_valid = parse_and_validate_coordinates(
    shelters, "避難所一覧"
)

## 住所→座標変換（Jageocoder）

座標が空欄で `address` 列がある行について、アップロード済みのJageocoder辞書で住所検索を行い座標を
補完します。既に座標がある行はそのまま使用します。外部の公開ジオコーディングサービスへは実住所を
送信しません。`ENABLE_GEOCODING = False`（上の「利用者設定」）の場合はこのセルは何もせずスキップします。

Jageocoderは住所の途中まで（町字・丁目程度）しか一致しない場合でも候補を返すことがあります。
街区・地番レベル以上まで一致しなかった行は **座標を自動採用せず**、`geocode_status = "coarse_match"`
として座標を空欄のまま残し、職員による住所確認の対象とします。なお一致レベルは「どこまで一致したか」
の目安であり、街区・地番レベルで一致した場合でも、入力した住所文字列全体が完全に一致したことまでは
保証しません。辞書検索中にエラーが発生した場合も `"not_found"` とは区別し、`"error"` として記録します。

In [ ]:
# Jageocoderの一致レベルの下限（この値未満は「粗い一致」とみなし、座標を自動採用せず確認対象とする）
# level はJageocoderの定義で 7=街区・地番(Block), 8=建物(Building) 等を示す値です
# （このNotebookの一次候補算出には、街区・地番レベル以上の座標で十分と考え7としています。
#  丁目程度までしか一致しなかった粗い結果を自動採用しないための下限値であり、
#  住所文字列全体が完全一致したことまでは保証しません）。
JAGEOCODER_MIN_LEVEL = 7


def init_jageocoder():
    import jageocoder

    if not JAGEOCODER_DB_DIR or not os.path.isdir(JAGEOCODER_DB_DIR):
        raise RuntimeError(
            "ENABLE_GEOCODING=True ですが、Jageocoderの辞書が見つかりません。"
            "「Jageocoder辞書ZIPのアップロード」で辞書ZIPを正しくアップロードしているか確認してください。"
        )
    try:
        jageocoder.init(db_dir=JAGEOCODER_DB_DIR)
    except Exception as error:
        raise RuntimeError(
            "Jageocoderのローカル辞書の初期化に失敗しました。"
            "「Jageocoder辞書ZIPのアップロード」で正しい辞書ZIPをアップロードしているか確認してください。"
            f" 詳細: {error}"
        )
    return jageocoder


def geocode_address(jageocoder_module, address):
    """Jageocoderのローカル辞書で住所を検索し、結果を状態付きの辞書で返す。

    status: "matched"(JAGEOCODER_MIN_LEVEL以上の一致・座標採用) / "coarse_match"(それより粗い一致・座標は未採用)
            / "not_found"(該当なし) / "error"(検索処理中の例外)
    座標を推測することはせず、一致しない・粗い一致の場合は座標を返さない。
    """
    if not isinstance(address, str) or not address.strip():
        return {"status": "not_found"}

    try:
        results = jageocoder_module.searchNode(address.strip())
    except Exception as error:
        return {"status": "error", "error": str(error)}

    if not results:
        return {"status": "not_found"}

    # jageocoderの戻り値の形状はバージョンにより差異があるため、取得できる範囲のみ利用する
    node = results[0].node
    try:
        matched_name = node.get_fullname()
        if isinstance(matched_name, (list, tuple)):
            matched_name = "".join(matched_name)
    except Exception:
        matched_name = str(node)
    level = getattr(node, "level", None)

    if level is not None and level < JAGEOCODER_MIN_LEVEL:
        return {"status": "coarse_match", "matched": matched_name, "level": level}

    return {
        "status": "matched",
        "lat": node.y,
        "lon": node.x,
        "matched": matched_name,
        "level": level,
    }


def fill_missing_coordinates(df, coord_valid, label):
    """座標が空欄の行についてのみ、address列からジオコーディングして座標を補完する。
    粗い一致・該当なし・エラーの行は座標を採用せず、geocode_statusに状態を残す。"""
    if not ENABLE_GEOCODING:
        return df, coord_valid

    df = df.copy()
    if "address" not in df.columns:
        print(f"[{label}] address列が無いため、ジオコーディングを行いません。")
        return df, coord_valid

    jageocoder_module = init_jageocoder()

    # 文字列(status/matched)を後から代入するため、float64ではなくobject dtypeで列を用意する
    df["geocode_status"] = pd.Series(np.nan, index=df.index, dtype="object")
    df["geocode_matched"] = pd.Series(np.nan, index=df.index, dtype="object")
    df["geocode_level"] = np.nan

    missing_mask = df["latitude"].isna() | df["longitude"].isna()
    target_count = int(missing_mask.sum())
    status_counts = {"matched": 0, "coarse_match": 0, "not_found": 0, "error": 0}

    for idx in df.index[missing_mask]:
        result = geocode_address(jageocoder_module, df.at[idx, "address"])
        status = result["status"]
        status_counts[status] = status_counts.get(status, 0) + 1
        df.at[idx, "geocode_status"] = status

        if status == "matched":
            df.at[idx, "latitude"] = result["lat"]
            df.at[idx, "longitude"] = result["lon"]
            df.at[idx, "geocode_matched"] = result["matched"]
            df.at[idx, "geocode_level"] = result["level"]
        elif status == "coarse_match":
            df.at[idx, "geocode_matched"] = result["matched"]
            df.at[idx, "geocode_level"] = result["level"]

    print(
        f"[{label}] ジオコーディング対象 {target_count}件: "
        f"詳細一致(座標採用) {status_counts['matched']}件 / "
        f"粗い一致(要確認) {status_counts['coarse_match']}件 / "
        f"該当なし {status_counts['not_found']}件 / "
        f"エラー {status_counts['error']}件"
    )

    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")
    valid = lat.notna() & lon.notna() & lat.between(-90, 90) & lon.between(-180, 180)
    df["latitude"], df["longitude"] = lat, lon
    return df, valid


residents, residents_coord_valid = fill_missing_coordinates(residents, residents_coord_valid, "要支援者一覧")
shelters, shelters_coord_valid = fill_missing_coordinates(shelters, shelters_coord_valid, "避難所一覧")

if not ENABLE_GEOCODING:
    print("ENABLE_GEOCODING=False のため、住所→座標変換をスキップしました。")

## ハザードデータの読込（任意）

`ENABLE_HAZARD_CHECK = True`（上の「利用者設定」）の場合のみ、ハザード区域のGeoJSONファイルを
アップロードします。洪水・土砂災害・津波・高潮等、複数種類のGeoJSONをまとめて選択できます。ファイル
ごとにハザード種別名を入力してください。座標系はWGS84（EPSG:4326）に統一され、Polygon/MultiPolygon
以外の空・不正なジオメトリは除外されます。`ENABLE_HAZARD_CHECK = False`（既定）の場合はこのセルは
スキップされ、ハザードデータなしで距離候補算出のみが実行されます。

**注意**: `ENABLE_HAZARD_CHECK = True` にした場合、有効なハザード区域ポリゴンが1件も読み込めなかったときは
「ハザードなし」とみなさず、ここで処理を停止します（判定していないことと、ハザード区域でないことを区別するためです）。

In [ ]:
def load_hazard_geojson(file_bytes, hazard_type, label):
    """GeoJSONを読み込み、hazard_type/geometryの2列に正規化し、WGS84(EPSG:4326)へ統一する。
    Polygon/MultiPolygon以外、または空・不正なジオメトリは除外する。"""
    gdf = gpd.read_file(io.BytesIO(file_bytes))
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    before = len(gdf)
    gdf = gdf[
        gdf.geometry.notna()
        & gdf.geometry.is_valid
        & gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    ]
    dropped = before - len(gdf)
    if dropped:
        print(f"  [{label}] 空・不正、またはPolygon/MultiPolygon以外のジオメトリを{dropped}件除外しました。")

    return gpd.GeoDataFrame(
        {"hazard_type": hazard_type, "geometry": gdf.geometry.values}, crs="EPSG:4326"
    )


hazard_gdf = None

if ENABLE_HAZARD_CHECK:
    print("ハザード区域のGeoJSONファイルをアップロードしてください（複数選択可）。")
    uploaded_hazards = files.upload()

    hazard_layers = []
    for hazard_filename, hazard_bytes in uploaded_hazards.items():
        hazard_type = input(
            f"'{hazard_filename}' のハザード種別名を入力してください（例: 洪水, 土砂災害, 津波, 高潮）: "
        ).strip()
        if not hazard_type:
            hazard_type = hazard_filename
        layer = load_hazard_geojson(hazard_bytes, hazard_type, hazard_filename)
        print(f"'{hazard_filename}' を hazard_type='{hazard_type}' として読み込みました。（有効{len(layer)}件）")
        hazard_layers.append(layer)

    if hazard_layers:
        hazard_gdf = gpd.GeoDataFrame(pd.concat(hazard_layers, ignore_index=True), crs="EPSG:4326")

    if hazard_gdf is None or len(hazard_gdf) == 0:
        raise RuntimeError(
            "ENABLE_HAZARD_CHECK=True ですが、有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
            "GeoJSONファイルが正しくアップロードされているか、ジオメトリ形式を確認してください。"
            "ハザード判定を行わない場合は、上の「利用者設定」で ENABLE_HAZARD_CHECK=False にしてください。"
        )

    print(f"ハザードデータを合計 {len(hazard_gdf)}件読み込みました。")
else:
    print("ENABLE_HAZARD_CHECK=False のため、ハザードデータの読込をスキップします。")

## 有効な避難所の確認

座標が不正な避難所の行は距離計算の対象から除外します。除外件数を表示し、
有効な避難所が0件の場合はここで処理を停止します（それ以外の場合は処理を継続します）。

In [ ]:
invalid_shelter_count = int((~shelters_coord_valid).sum())
shelters_valid = shelters.loc[shelters_coord_valid].reset_index(drop=True)

print(f"避難所一覧: 全{len(shelters)}件中、座標が不正なため {invalid_shelter_count}件を除外しました。")
print(f"距離計算に使用する有効な避難所: {len(shelters_valid)}件")

if len(shelters_valid) == 0:
    raise RuntimeError(
        "有効な座標を持つ避難所が0件です。避難所一覧CSVの latitude / longitude 列を確認してください。"
    )

## 避難所候補の距離計算

各要支援者について、有効な避難所すべてとの直線距離（`geopy.distance.geodesic`、メートル単位）を計算し、
近い順に `TOP_N` 件を候補として算出する関数を定義します。これは道路距離ではありません。
並び替えは丸める前の距離で行い、メートル単位への丸め（小数1桁）はCSVに出力する値を作成する際にのみ行います。
距離が同一の場合でも結果順が実行ごとにばらつかないよう、避難所名を用いて順序を安定させます。
避難所一覧に災害種別列（`災害種別_〜`）があった場合は、その避難所の対応区分（対応済み／2階以上であれば対応済み等）
も候補情報として保持します（距離順位には影響しません）。

In [ ]:
def compute_candidates(resident_lat, resident_lon, shelters_df, top_n):
    """要支援者の座標から近い順に避難所候補を [(名前, 距離m, 緯度, 経度, 災害種別対応区分), ...] で返す。
    座標が欠損、または緯度・経度が有効範囲外であればNoneを返す（geodesic()に不正値を渡さない）。
    距離は丸めずに返す（並び替え後、出力時にのみ丸める）。対応区分は災害種別列が無ければNoneのまま。"""
    if not is_valid_coordinate(resident_lat, resident_lon):
        return None

    has_disaster_info = "_disaster_support" in shelters_df.columns
    resident_coord = (resident_lat, resident_lon)
    records = []
    for _, shelter in shelters_df.iterrows():
        shelter_coord = (shelter["latitude"], shelter["longitude"])
        distance_m = geodesic(resident_coord, shelter_coord).meters
        disaster_support = shelter["_disaster_support"] if has_disaster_info else np.nan
        records.append(
            (shelter["name"], distance_m, shelter["latitude"], shelter["longitude"], disaster_support)
        )

    records.sort(key=lambda record: (record[1], record[0]))
    return records[:top_n]

## ハザード区域判定用の関数

要支援者地点・候補避難所地点がハザード区域の内部または境界上にあるか、
また要支援者と候補避難所を結ぶ直線がハザード区域と交差するかを判定する関数を定義します。

直線交差の判定は **道路上の避難経路判定ではありません**。単純に2地点を結んだ直線上にハザード区域が
存在するかどうかを確認する参考情報です。要支援者自身がハザード区域内にいる場合、直線は始点で
既に区域と交差しますが、「地点の判定」と「直線の判定」は別の列として出力するため、混同しないでください。

In [ ]:
def hazard_types_at_point(lat, lon, hazard_area):
    """座標がハザード区域の内部または境界上にあるかどうかと、該当するhazard_type（;区切り）を返す。
    座標が欠損・範囲外の場合は判定不能としてnp.nanを返す（区域外=Falseと混同しない）。"""
    if hazard_area is None or len(hazard_area) == 0:
        return False, ""
    if not is_valid_coordinate(lat, lon):
        return np.nan, np.nan

    point = Point(lon, lat)
    hit_types = sorted(hazard_area.loc[hazard_area.geometry.intersects(point), "hazard_type"].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


def hazard_types_on_line(lat1, lon1, lat2, lon2, hazard_area):
    """2地点を結ぶ直線がハザード区域と交差するかどうかと、該当するhazard_typeを返す（道路経路上の判定ではない）。
    いずれかの座標が欠損・範囲外の場合は判定不能としてnp.nanを返す。"""
    if hazard_area is None or len(hazard_area) == 0:
        return False, ""
    if not is_valid_coordinate(lat1, lon1) or not is_valid_coordinate(lat2, lon2):
        return np.nan, np.nan

    line = LineString([(lon1, lat1), (lon2, lat2)])
    hit_types = sorted(hazard_area.loc[hazard_area.geometry.intersects(line), "hazard_type"].unique())
    return (len(hit_types) > 0), ";".join(hit_types)

## 候補算出とハザード判定の実行

要支援者ごとに、避難所候補・距離・（`ENABLE_HAZARD_CHECK=True` の場合のみ）ハザード判定結果を組み立てます。
距離による候補順位はハザード判定結果によって変更されません。座標がない・不正な要支援者の行も削除せず、
候補・距離を空欄のまま保持します（`match_status` 列で `ok` / `no_coordinates`(座標欄が空欄) /
`invalid_coordinates`(値はあるが範囲外) の状態を確認できます）。
避難所一覧に災害種別列があった場合は `candidate_N_disaster_support` 列に、災害種別ごとの対応区分
（例: `洪水:対応済み;高潮:2階以上であれば対応済み`）を出力します（対応区分によって候補の順位変更・自動除外は行いません）。

In [ ]:
result_records = []

for _, resident in residents.iterrows():
    lat, lon = resident["latitude"], resident["longitude"]
    candidates = compute_candidates(lat, lon, shelters_valid, TOP_N)

    if pd.isna(lat) or pd.isna(lon):
        coord_status = "no_coordinates"
    elif not is_valid_coordinate(lat, lon):
        coord_status = "invalid_coordinates"
    else:
        coord_status = "ok"

    record = {}

    if ENABLE_HAZARD_CHECK:
        in_hazard, hazard_types = hazard_types_at_point(lat, lon, hazard_gdf)
        record["resident_in_hazard"] = in_hazard
        record["resident_hazard_types"] = hazard_types

    for i in range(TOP_N):
        n = i + 1
        candidate_col = f"candidate_{n}"
        distance_col = f"distance_{n}_m"
        disaster_col = f"candidate_{n}_disaster_support"
        shelter_hazard_col = f"candidate_{n}_shelter_in_hazard"
        shelter_hazard_types_col = f"candidate_{n}_shelter_hazard_types"
        line_hazard_col = f"candidate_{n}_straight_line_intersects_hazard"
        line_hazard_types_col = f"candidate_{n}_straight_line_hazard_types"

        if candidates is not None and i < len(candidates):
            shelter_name, distance_m, shelter_lat, shelter_lon, disaster_support = candidates[i]
            record[candidate_col] = shelter_name
            record[distance_col] = round(distance_m, 1)

            if HAS_DISASTER_TYPE_COLUMNS:
                record[disaster_col] = disaster_support

            if ENABLE_HAZARD_CHECK:
                shelter_in_hazard, shelter_hazard_types = hazard_types_at_point(
                    shelter_lat, shelter_lon, hazard_gdf
                )
                record[shelter_hazard_col] = shelter_in_hazard
                record[shelter_hazard_types_col] = shelter_hazard_types

                line_intersects, line_hazard_types = hazard_types_on_line(
                    lat, lon, shelter_lat, shelter_lon, hazard_gdf
                )
                record[line_hazard_col] = line_intersects
                record[line_hazard_types_col] = line_hazard_types
        else:
            record[candidate_col] = np.nan
            record[distance_col] = np.nan
            if HAS_DISASTER_TYPE_COLUMNS:
                record[disaster_col] = np.nan
            if ENABLE_HAZARD_CHECK:
                record[shelter_hazard_col] = np.nan
                record[shelter_hazard_types_col] = np.nan
                record[line_hazard_col] = np.nan
                record[line_hazard_types_col] = np.nan

    record["match_status"] = coord_status
    result_records.append(record)

results_df = pd.DataFrame(result_records)
final_df = pd.concat([residents.reset_index(drop=True), results_df], axis=1)

print("候補算出が完了しました。")

## 結果確認

CSVを出力する前に、Notebook上で処理結果の概要と先頭数行を確認します。

In [ ]:
total_residents = len(final_df)
ok_count = int((final_df["match_status"] == "ok").sum())
no_coord_count = int((final_df["match_status"] == "no_coordinates").sum())
invalid_coord_count = int((final_df["match_status"] == "invalid_coordinates").sum())

print(f"要支援者件数: {total_residents}件")
print(f"距離計算できた件数: {ok_count}件")
print(f"座標が空欄のため距離計算できなかった件数: {no_coord_count}件")
print(f"座標が範囲外で不正なため距離計算できなかった件数: {invalid_coord_count}件")
print(f"距離計算に使用した有効な避難所件数: {len(shelters_valid)}件")

if ENABLE_GEOCODING and "geocode_status" in final_df.columns:
    print("要支援者一覧のジオコーディング結果:")
    for status, count in final_df["geocode_status"].value_counts(dropna=True).items():
        print(f"  {status}: {count}件")

if ENABLE_HAZARD_CHECK:
    resident_hazard_count = int((final_df["resident_in_hazard"] == True).sum())
    print(f"ハザード区域内（境界上含む）にいる要支援者数: {resident_hazard_count}件")

    shelter_hazard_flags = [
        final_df[f"candidate_{i + 1}_shelter_in_hazard"] == True for i in range(TOP_N)
    ]
    line_hazard_flags = [
        final_df[f"candidate_{i + 1}_straight_line_intersects_hazard"] == True for i in range(TOP_N)
    ]
    shelter_hazard_count = int(pd.concat(shelter_hazard_flags, axis=1).sum().sum())
    line_hazard_count = int(pd.concat(line_hazard_flags, axis=1).sum().sum())

    print(f"ハザード区域内にある候補避難所の件数（延べ、TOP_N分の合計）: {shelter_hazard_count}件")
    print(f"候補避難所への直線がハザード区域と交差する件数（延べ、TOP_N分の合計）: {line_hazard_count}件")

display(final_df.head())

## CSV出力・ダウンロード

結果をExcelで文字化けしにくい `utf-8-sig`（UTF-8 BOM付き）でCSVに出力し、ブラウザへダウンロードします。
出力CSVには個人情報が含まれ得るため、取り扱いに注意してください。

In [ ]:
OUTPUT_FILENAME = "assigned_shelters.csv"

final_df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")
print(f"'{OUTPUT_FILENAME}' を出力しました。")

files.download(OUTPUT_FILENAME)